# 05_04 Running fair experiments

*📖 Not in the book*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → [model learns] → model writes → MCP tool
> ```
>
> You are here: **model learns**
>
> Project files this connects to:
>
> - `src/homer_ai/train_homer.py`
>
> **This notebook:** We change a line of code or a setting, and the loss moves. How do we KNOW our change did it, and not luck or a hidden mistake?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How to save a baseline, a saved "before" result
- How to compare before and after, and spot a bug or a refactor
- Why the seed stays fixed, and why you change one thing at a time
- How to time a run, and how to keep a log of results

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **baseline** | a saved "before" result, kept so every change can be compared with it | `baseline.txt` |
| **standard output (stdout)** | the normal channel a program prints to; `print()` uses it | `print("Epoch 100/400 …")` |
| **standard error (stderr)** | a second channel, used for warnings and errors | a NumPy warning |
| **redirect (`>`)** | in the terminal, `>` sends stdout into a file instead of the screen | `… > baseline.txt` |
| **`diff`** | a terminal command that compares two files and prints only the lines that differ; no output means identical | `diff baseline.txt after.txt` |
| **refactor** | reorganising code without changing what it does | moving the model into `model.py` |
| **results log** | a table where you write down each run's settings and result | run, what changed, final loss |

## 3. The idea

### ❓ I changed something and the loss moved. How do I know my change did it?

Only by comparing it with a run where nothing else changed. That's a small experiment, and it comes down to four habits:

```
1. save a BASELINE        the "before": one run, written to a file
2. fix the SEED           so luck is the same in both runs
3. change ONE thing       so you know what to blame
4. COMPARE and LOG        diff the files; write the result in a table
```

You've already done it once. The four runs of [05_02](05_02_learning_rate.ipynb) jumped around, and they only made sense because each one changed one thing, with the same seed, and we kept every number.

*In my day job I build [evaluation dashboards](https://icodenet.github.io/eval-dashboards/docs/) for AI agents. The habit starts right here: one baseline, change one thing, compare.*

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ What's a baseline, and how do I save one?

A **baseline** is a saved "before". Before you change anything, run the training once and keep what it printed in a file:

```
uv run python src/homer_ai/train_homer.py > baseline.txt
```

(Don't run it yet. It takes minutes and it writes a model file. The next question says what to do first.)

`>` is a **redirect**. It sends the normal output (stdout) into the file and not to the screen. Warnings travel on a second channel, stderr, which `>` doesn't catch, so they still show up on the screen. That's why the file holds only our clean lines.

👉 **Predict first:** if a program prints one normal line and one warning, which one will `result.stdout` hold?

In [ ]:
import subprocess
import sys

script = 'import sys\nprint("normal output")\nprint("a warning", file=sys.stderr)'
result = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True)
print("stdout (what > catches):", repr(result.stdout))
print("stderr (what > misses): ", repr(result.stderr))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The two lines went to two different channels. `subprocess.run` starts the small script, and `capture_output=True` catches both channels so we can look at them. (`repr()` shows the hidden `\n` at the end of each line, as in [02_04](../02_text_to_numbers/02_04_sliding_window.ipynb).)

</details>

### ❓ What does a baseline file look like?

Eighteen lines: one "Training on…" line, sixteen loss lines (one every 50 epochs, for 800 epochs), and one "Saved…" line. The loss is the average per chunk, and `lr` is the learning rate sliding down its cosine curve ([05_02](05_02_learning_rate.ipynb)):

```
baseline.txt
 1  Training on 19701 Greek characters (Vocab size: 137)...
 2  Epoch 50/800 | Avg loss: … | lr: 0.00990
 3  Epoch 100/800 | Avg loss: … | lr: 0.00962
    …
17  Epoch 800/800 | Avg loss: 2.3817 | lr: 0.00000
18  Saved trained model to …/src/homer_ai/homer_model.pt
```

(The `…` are losses we haven't copied here. The last one, 2.3817, is what our run printed. Yours should match on the same machine and PyTorch version, and may differ in the last digits on another.)

**Two things to know before you run that command.** It takes about 8 to 10 minutes on a laptop. And line 18 is real: the run **writes `src/homer_ai/homer_model.pt`, and replaces the one that's there** if you trained before. The script has no option for another file name. So if you want to keep your current model, copy it first:

```
cp src/homer_ai/homer_model.pt my_model_before.pt
```

(The ready-made files in `models/` are never touched. After the run, the notebooks say `Using your own model`.)

You don't need to run it now. Here's the same idea in a few seconds. `tiny_train` is the small training run you built in [05_01](05_01_loss_and_epochs.ipynb) (5 lines of Book 1, 24-character chunks). It's kept in `src/homer_ai/lessons.py`, so we import it, and `to_log` turns its losses into lines shaped like the real printout. It saves nothing. The "Saved" line is just text.

👉 **Predict first:** with 100 epochs, printing every 25, how many lines will the log have?

In [ ]:
import torch
import torch.nn.functional as F

from homer_ai.lessons import practice_data, tiny_train
from homer_ai.model import HomerGreekLLM

# the practice text of 05_01: the first 5 lines of Book 1, as text and as numbers
practice_text, tokenizer, data = practice_data()


def to_log(losses, every=25):
    """Turn a list of average losses into lines like the real printout."""
    lines = [f"Training on {len(data)} Greek characters..."]
    for epoch in range(every, len(losses) + 1, every):
        lines.append(f"Epoch {epoch}/{len(losses)} | Avg loss: {losses[epoch - 1]:.4f}")
    lines.append("Saved trained model to homer_model.pt")
    return lines


baseline_losses = tiny_train(seed=42, epochs=100)
baseline = to_log(baseline_losses)
print(len(baseline), "lines")
for line in baseline:
    print(line)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

6 lines: 1 + 4 + 1, the same shape as the real baseline, only shorter. Here a **list of lines** in Python plays the role of `baseline.txt`.

</details>

### ❓ How do I compare the new run with the baseline?

In the terminal, with **`diff`**:

```
uv run python src/homer_ai/train_homer.py > after.txt
diff baseline.txt after.txt
```

`diff` prints only the lines that differ. So no output means the two files are identical. That's exactly what you want after a **refactor** (reorganising code without changing what it does). Say you move the model into its own file, `model.py`, and `diff` prints nothing: then nothing broke. It only works because of the fixed seed. Without it, even identical code would give different numbers.

In Python, you compare two lists of lines with `==`, and the standard-library module `difflib` prints a readable diff (`unified_diff` marks removed lines with `-` and added lines with `+`).

👉 **Predict first:** we run the same code again, with the same seed. What will `==` say, and what will `unified_diff` print?

In [ ]:
import difflib

after = to_log(tiny_train(seed=42, epochs=100))
print("identical?", after == baseline)
print("diff lines:", list(difflib.unified_diff(baseline, after, lineterm="")))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`True`, and an empty list. No differences. In the terminal, this is `diff` printing nothing.

</details>

### ❓ And what does a real bug look like in a diff?

Here's one we made ourselves. We indented the three training lines one level too little. Indentation decides which lines belong to the loop, so those lines fell out of the `for` loop. They ran once per epoch, after the loop, and the model learned from only **one chunk of nine**. The code still ran without errors. Only the diff showed that something had changed.

```
correct                                  one level too little (bug)
for i in range(...):                     for i in range(...):
    loss = ...                               loss = ...
    optimizer.zero_grad()                optimizer.zero_grad()   <- outside the loop
    loss.backward()                      loss.backward()
    optimizer.step()                     optimizer.step()
```

Here's the same mistake, run on the tiny text.

👉 **Predict first:** which lines of the log will differ: the first, the middle, the last?

In [ ]:
def tiny_train_bug(seed, epochs, lr=0.01, chunk_size=24):
    # The same as tiny_train, but the three training lines are indented one
    # level too little: they sit AFTER the for loop, not inside it.
    torch.manual_seed(seed)
    model = HomerGreekLLM(vocab_size=tokenizer.vocab_size, embed_dim=32)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    average_losses = []
    for _epoch in range(epochs):
        losses = []
        for i in range(0, len(data) - chunk_size, chunk_size):
            inputs = data[i : i + chunk_size]
            targets = data[i + 1 : i + chunk_size + 1]
            loss = F.cross_entropy(model(inputs), targets)
            losses.append(loss.item())
        optimizer.zero_grad()  # runs once per epoch: only the LAST chunk teaches
        loss.backward()
        optimizer.step()
        average_losses.append(sum(losses) / len(losses))
    return average_losses


buggy = to_log(tiny_train_bug(42, 100))
for line in difflib.unified_diff(baseline, buggy, "baseline", "buggy", lineterm=""):
    print(line)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The first and last lines are the same (`Training on…`, `Saved…`), and only the loss lines changed: `-` is the baseline, `+` is the buggy run. And look at the buggy numbers: they **climb**, from 7.7 to 14.2. That's far worse than plain guessing, which would score about 3.95 here (52 characters, see [05_01](05_01_loss_and_epochs.ipynb)). Why? The model now learns from the last chunk only, epoch after epoch. So it gets very sure about that one chunk, and takes the same sureness to the other eight, where it's wrong. Being confidently wrong is what costs the most (05_01 again), so the average goes up while the model "learns". So a change you didn't mean to make leaves a fingerprint. **A diff turns a silent bug into a visible one.**

</details>

### ❓ Why must the seed stay fixed?

The **seed** (see [05_01](05_01_loss_and_epochs.ipynb)) fixes the starting point of the "random" numbers, like shuffling a deck of cards the same way every time. `torch.manual_seed(42)` comes before the model is made, so the model starts with the same weights. If the seed changes too, you can't tell a real effect from plain luck.

One more rule: you only get the same numbers from the same seed plus the **same order of random steps**. If you add code that draws one extra random number before the model is made, the whole shuffled deck shifts.

👉 **Predict first:** which of these three will match the first one?

In [ ]:
torch.manual_seed(42)
first = torch.rand(3)

torch.manual_seed(42)
again = torch.rand(3)

torch.manual_seed(42)
_ = torch.rand(1)  # one extra random step
shifted = torch.rand(3)

print("first:  ", first)
print("again:  ", again)
print("shifted:", shifted)
print("first == again?  ", bool((first == again).all()))
print("first == shifted?", bool((first == shifted).all()))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The same seed and the same steps give exactly the same numbers. One extra random step in between, and everything shifted. (`torch.rand(3)` draws 3 random numbers between 0 and 1.)

</details>

### ❓ Why change only one thing at a time?

Because if two things change and the result improves, you can't say which one helped. That's how the runs of [05_02](05_02_learning_rate.ipynb) were set up: first only the **learning rate** changed (0.01 → 0.003), then only the schedule (constant → cosine), then only the epochs (400 → 800). Each change gave its own answer.

Here's a tiny version, using the picture from the start: the same seed, one change. `baseline_losses` is the lr 0.01 run from above. The next run changes only the learning rate.

👉 **Predict first:** with 100 epochs, will the smaller learning rate end with a lower or a higher loss?

In [ ]:
slow_losses = tiny_train(seed=42, epochs=100, lr=0.003)  # the ONE change

print(f"lr 0.01  (baseline): final loss {baseline_losses[-1]:.3f}")
print(f"lr 0.003 (one change): final loss {slow_losses[-1]:.3f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The smaller learning rate ended higher. Smaller steps get less far in the same number of epochs, which is the trade-off from [05_02](05_02_learning_rate.ipynb). And we know the learning rate is to blame, because nothing else changed.

</details>

### ❓ Can I see the two runs together?

Let's draw both loss curves. The orange line is the baseline. We also draw a second baseline run (same seed) as white dots on top of it. If the dots sit exactly on the line, the two runs were identical, so any gap to the grey line comes from the one change.

👉 **Predict first:** will you see the white dots as a separate line?

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import ACCENT, INK, MUTED, studio_axes, studio_figure

again_losses = tiny_train(seed=42, epochs=100)  # the baseline, run a second time
epochs = list(range(1, 101))

fig = studio_figure("Same seed, one change", "Only the learning rate is different")
ax = studio_axes(fig)
ax.plot(epochs, baseline_losses, color=ACCENT, linewidth=3)
ax.plot(epochs[::4], again_losses[::4], "o", color=INK, markersize=5)
ax.plot(epochs, slow_losses, color=MUTED, linewidth=3)
ax.text(
    40, 0.4, "lr 0.01 (baseline)",
    color=ACCENT, family="monospace", fontweight="bold",
)
ax.text(
    30, 2.4, "lr 0.003 (the one change)",
    color=MUTED, family="monospace", fontweight="bold",
)
ax.set_ylim(0.25, 4.2)
ax.text(
    98, 0.97 * max(baseline_losses[0], slow_losses[0]),
    "white dots: the baseline run again,\nsame seed: exactly on the line",
    color=INK, ha="right", va="top", family="monospace",
)
ax.set_xlabel("epoch")
ax.set_ylabel("average loss per chunk")
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The white dots sit exactly on the orange line (the same seed repeats a run exactly), and the grey line is the one change. So that gap is the effect of the learning rate, and nothing else.

</details>

### ❓ What goes wrong if I change two things at once?

Let's try it. We change the learning rate **and** double the epochs in one run, and compare it with two runs that each change one thing.

👉 **Predict first:** from the "both" number alone, will you be able to tell which change helped?

In [ ]:
more_epochs = tiny_train(seed=42, epochs=200)  # ONE change: epochs
both = tiny_train(seed=42, epochs=200, lr=0.003)  # TWO changes at once

print(f"baseline  (lr 0.01,  100 epochs):  {baseline_losses[-1]:.3f}")
print(f"lr only   (lr 0.003, 100 epochs):  {slow_losses[-1]:.3f}")
print(f"epochs only (lr 0.01,  200 epochs): {more_epochs[-1]:.3f}")
print(f"both      (lr 0.003, 200 epochs):  {both[-1]:.3f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The "both" run ends somewhere in between, and that number alone can't tell you why. More epochs help, a smaller learning rate hurts, and the two effects get mixed into one number. Only the two single-change runs, each compared with the baseline, can untangle them.

</details>

### ❓ How do I time a run?

Put `time` in front of the command, and throw the output away with `/dev/null` (a special "file" that swallows everything), so that printing doesn't slow the clock:

```
time uv run python src/homer_ai/train_homer.py > /dev/null
```

Read the line that starts with **`real`**. That's the time on a wall clock, start to finish. A full Book 1 run takes about 8 to 10 minutes on a laptop (one of ours: `real 7m47.27s`). And remember, it writes `homer_model.pt`.

In Python, `time.perf_counter()` gives a clock reading in seconds. Take one before and one after, and subtract.

👉 **Predict first:** 40 epochs instead of 20. About how long compared with 20?

In [ ]:
import time

for epochs in [20, 40]:
    start = time.perf_counter()
    tiny_train(seed=42, epochs=epochs)
    seconds = time.perf_counter() - start
    print(f"{epochs} epochs: {seconds:.2f} seconds")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Twice the epochs takes about twice as long. That's why timing matters: you know what an idea will cost before you commit to the 800-epoch run.

</details>

### ❓ Where do I write all this down?

In a **results log**: one row per run, with what you changed and what came out. This is the log of the four runs of [05_02](05_02_learning_rate.ipynb) (the final loss is the average per chunk, as printed there):

| # | The one thing that changed | Learning rate | Epochs | Final loss |
|---|---|---|---|---|
| 1 | nothing: the baseline | 0.01 | 400 | 0.475 |
| 2 | only the learning rate (against 1) | 0.003 | 400 | 0.135 |
| 3 | only the schedule: cosine (against 1) | 0.01 → 0 | 400 | 0.067 |
| 4 | only the epochs (against 3) | 0.01 → 0 | 800 | 0.008 |

And here's the log for our tiny runs, built from the numbers computed above.

👉 **Predict first:** which row has the lowest final loss?

In [ ]:
log = [
    ("baseline", 0.01, 100, baseline_losses[-1]),
    ("lr only", 0.003, 100, slow_losses[-1]),
    ("epochs only", 0.01, 200, more_epochs[-1]),
    ("both (unfair)", 0.003, 200, both[-1]),
]

print(f"{'run':<15}{'lr':>8}{'epochs':>8}{'final loss':>12}")
for name, lr, epochs, final in log:
    print(f"{name:<15}{lr:>8}{epochs:>8}{final:>12.3f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A log you can still read in ten seconds a month from today. Each row says what changed, so you can trust it. And the "both" row is marked unfair on purpose.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Is one run enough to trust?

For comparing two versions of the same code with a fixed seed, yes: any difference comes from your change. But a different seed might rank two ideas differently. So for a big decision, repeat the comparison with a few seeds.

### ❓ Do I have to use `diff` and `time` in the terminal?

They're the quickest way, and you'll see them everywhere in AI engineering. The notebook versions here (`==`, `difflib`, `time.perf_counter()`) give you the same answers inside Python.

## 5. Recap

- Save a baseline, change one thing, keep the seed fixed, and compare (`diff`). Then you know what a change did.
- A refactor leaves the diff empty. A bug leaves a fingerprint.
- Time your runs, and write every run in a results log.

**You can now:**

- run an experiment that changes one thing, with a fixed seed and a saved baseline
- tell a refactor from a bug by comparing the losses

**Next:** [05_05: Validation](05_05_validation.ipynb): the real split, and the validation loss.